In [1]:
import pandas as pd
import numpy as np
import os

base_path = r'D:\datasets\match review' 
folders = ['ODI', 'T20', 'Test']
all_data = []

# Added 3rd and 4th innings to the required columns
required_cols = {
    'Date': ['Date'],
    'Series': ['Series'],
    'Venue': ['Full Venue Name', 'Full Venue', 'Venue'],
    'Toss_Winner': ['Toss Winner', 'Toss Winn'],
    'Toss_Decision': ['Toss Decis', 'Toss Decision'],
    '1st_Inn_Runs': ['1st Inn Runs', '1st Inn Ru', '1st Inn Rur'],
    '1st_Inn_Wkts': ['1st Inn Wk', '1st Inn Wkts'],
    '2nd_Inn_Runs': ['2nd Inn Ru', '2nd Inn Runs'],
    '2nd_Inn_Wkts': ['2nd Inn Wk', '2nd_Inn_Wkts'],
    '3rd_Inn_Runs': ['3rd Inn Ru', '3rd Inn Runs', '3rd_Inn_Runs'], # Added
    '3rd_Inn_Wkts': ['3rd Inn Wk', '3rd_Inn_Wkts'],               # Added
    '4th_Inn_Runs': ['4th Inn Ru', '4th Inn Runs', '4th_Inn_Runs'], # Added
    '4th_Inn_Wkts': ['4th Inn Wk', '4th_Inn_Wkts'],               # Added
    'Result': ['Result'],
    'POM': ['Player of the Match', 'Player of t', 'POM']
}

for folder in folders:
    f_path = os.path.join(base_path, folder)
    if os.path.exists(f_path):
        for file in os.listdir(f_path):
            if file.endswith('.csv'):
                temp_df = pd.read_csv(os.path.join(f_path, file))
                temp_df.columns = temp_df.columns.str.strip()
                
                new_df = pd.DataFrame()
                for standard, aliases in required_cols.items():
                    actual_col = next((c for c in aliases if c in temp_df.columns), None)
                    new_df[standard] = temp_df[actual_col] if actual_col else 0
                
                new_df['Format'] = folder
                all_data.append(new_df)

df_master = pd.concat(all_data, ignore_index=True)
print("Data Load complete with 3rd/4th innings support.")

Data Load complete with 3rd/4th innings support.


In [4]:
from sklearn.preprocessing import LabelEncoder

# 1. Numeric Polish
score_cols = ['1st_Inn_Runs', '1st_Inn_Wkts', '2nd_Inn_Runs', '2nd_Inn_Wkts', 
              '3rd_Inn_Runs', '3rd_Inn_Wkts', '4th_Inn_Runs', '4th_Inn_Wkts']

for col in score_cols:
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce').fillna(0).astype(int)

# 2. Search Field for the flexible logic we built
df_master['Search_Field'] = (
    df_master['Series'].astype(str) + " " + 
    df_master['Toss_Winner'].astype(str) + " " + 
    df_master['Result'].astype(str)
).str.upper()

# 3. Define AI Variables (X, y)
le_team = LabelEncoder()
le_venue = LabelEncoder()

# Create encoded versions of text data so the model can read them
df_master['Toss_Enc'] = le_team.fit_transform(df_master['Toss_Winner'].astype(str))
df_master['Venue_Enc'] = le_venue.fit_transform(df_master['Venue'].astype(str))

# X = Input data | y = What we want to predict
X = df_master[['Toss_Enc', 'Venue_Enc', '1st_Inn_Wkts']]
y_reg = df_master['1st_Inn_Runs']
y_cls = df_master.apply(lambda x: 1 if str(x['Toss_Winner']) in str(x['Result']) else 0, axis=1)

print("Numeric conversion ready and AI variables (X, y) defined.")

Numeric conversion ready and AI variables (X, y) defined.


In [5]:
import xgboost as xgb
reg_model = xgb.XGBRegressor(n_estimators=50).fit(X, y_reg)
cls_model = xgb.XGBClassifier(n_estimators=50).fit(X, y_cls)
print("Models Trained.")

Models Trained.


In [9]:
def generate_custom_review(team_a, team_b, match_format='T20', match_date=None):
    team_a, team_b = team_a.upper(), team_b.upper()
    fmt_upper = match_format.upper()
    
    # 1. Flexible Search Logic
    mask = (df_master['Format'].str.upper() == fmt_upper)
    
    if match_date:
        mask = mask & (df_master['Date'].astype(str).str.contains(match_date))
        mask = mask & (
            (df_master['Search_Field'].str.contains(team_a)) | 
            (df_master['Search_Field'].str.contains(team_b))
        )
    else:
        mask = mask & (
            (df_master['Search_Field'].str.contains(team_a)) & 
            (df_master['Search_Field'].str.contains(team_b))
        )
        
    matches = df_master[mask]
    if matches.empty: 
        return f"Could not locate a match for {team_a} vs {team_b} on the specified parameters."
    
    m = matches.iloc[-1]
    
    # --- BUILD PROFESSIONAL OUTPUT ---
    res = f"# Match Report: {m['Series']} ({m['Format']})\n"
    res += f"Date:{m['Date']}\n"
    res += f"The match was played at the historic {m['Venue']}, providing an electric atmosphere for this encounter.\n"
    res += f"---\n"
    
    # Professional Toss Summary
    res += f"The action commenced when {m['Toss_Winner']} won the toss and made the tactical decision to {m['Toss_Decision']} first.\n\n"
    
    # Innings Display
    res += "# Scorecard Summary\n"
    res += f" 1st Innings: {m['1st_Inn_Runs']} runs for the loss of {m['1st_Inn_Wkts']} wickets.\n"
    
    if m['2nd_Inn_Runs'] > 0:
        res += f" 2nd Innings: {m['2nd_Inn_Runs']} runs for the loss of {m['2nd_Inn_Wkts']} wickets.\n"
    
    # Test Match Specifics
    if fmt_upper == 'TEST':
        if m['3rd_Inn_Runs'] > 0:
            res += f" 3rd Innings: {m['3rd_Inn_Runs']} runs for the loss of {m['3rd_Inn_Wkts']} wickets.\n"
        if m['4th_Inn_Runs'] > 0:
            res += f" 4th Innings: {m['4th_Inn_Runs']} runs for the loss of {m['4th_Inn_Wkts']} wickets.\n"
    
    # Result Sentence Formatting
    raw_res = str(m['Result'])
    if "(" in raw_res:
        team_win = raw_res.split("(")[0].strip()
        margin = raw_res.split("(")[1].replace(")", "").strip()
        final_res_string = f"After a hard-fought contest, {team_win} emerged victorious, winning the match by {margin}."
    else:
        final_res_string = f"The match concluded with the following result: {raw_res}."

    res += f"\n{final_res_string}\n"
    
    # Player of the Match Logic: Hide for TEST, Show for others
    if fmt_upper != 'TEST':
        pom = m['POM']
        if pom != 0 and str(pom) != '0' and str(pom).lower() != 'nan':
            res += f"\nFor an outstanding individual performance, {pom} was named the Player of the Match."
    
    return res

# Test it with the England vs Scotland T20 match
print(generate_custom_review("England", "ind", "T20", match_date="26/03/05"))

# Match Report: T20 WC SF (ENG) (T20)
Date:26/03/05
The match was played at the historic Wankhede Stadium, providing an electric atmosphere for this encounter.
---
The action commenced when India won the toss and made the tactical decision to Bowl first.

# Scorecard Summary
 1st Innings: 174 runs for the loss of 6 wickets.
 2nd Innings: 167 runs for the loss of 8 wickets.

After a hard-fought contest, IND emerged victorious, winning the match by 7 runs.

For an outstanding individual performance, Sanju Samson was named the Player of the Match.


In [7]:
# Search for Scotland and the Tournament name since England isn't listed in the row
print(generate_custom_review("SCO", "T20", "T20", match_date="24/06/04"))

# Match Report: T20 WC (SCO) (T20)
Date:24/06/04
The match was played at the historic Kensington Oval, providing an electric atmosphere for this encounter.
---
The action commenced when Scotland won the toss and made the tactical decision to Bat first.

# Scorecard Summary
 1st Innings: 90 runs for the loss of 0 wickets.

The match concluded with the following result: No Result.



In [10]:
import joblib

# Define the paths based on your folder structure
PROCESSED_DATA_PATH = "../data/processed/"
MODEL_DIR_PATH = "../models/"

# 1. Save the Match Review Master Data (For the lookup function)
df_master.to_csv(f"{PROCESSED_DATA_PATH}review_master.csv", index=False)

# 2. Save the Match Review Models (XGBoost)
joblib.dump(reg_model, f"{MODEL_DIR_PATH}review_reg_model.pkl")
joblib.dump(cls_model, f"{MODEL_DIR_PATH}review_cls_model.pkl")

print(f"Match Review data saved to: {PROCESSED_DATA_PATH}")
print(f"Match Review models saved to: {MODEL_DIR_PATH}")

Match Review data saved to: ../data/processed/
Match Review models saved to: ../models/
